# Bases de données — Exercice 1 : création d'une table et import des données (correction)

Ce notebook contient, pour chaque question, trois blocs **toujours séparés** :

| Bloc | Contenu |
|:---|:---|
| 📝 **ÉNONCÉ** | la question, recopiée du sujet |
| ✅ **RÉSOLUTION** | la réponse attendue (requête SQL, code et/ou texte) |
| 💡 **EXPLICATION PÉDAGOGIQUE** | pourquoi ça marche, le lien avec le cours, les erreurs à éviter. **N'est pas à recopier.** |

**Notions travaillées :** `CREATE TABLE`, types (domaines de valeurs), `INSERT INTO`, `SELECT *`, générer un script SQL avec Python, lire un fichier CSV, échapper une apostrophe.

**Données :** le fichier `countries.csv` n'est pas encore disponible sur le site du professeur. Le fichier `countries.csv` placé à côté de ce notebook a été **reconstitué** à partir de la table `Countries` de l'exercice 2 (mêmes pays, mêmes identifiants), avec les colonnes `id`, `code`, `name`, `continent`. Le script de la question 2 (b) fonctionne aussi si le vrai fichier contient des colonnes supplémentaires.

Exécuter les cellules **dans l'ordre** (*Cell → Run All*).

---
# MISE EN PLACE

En classe, les requêtes sont tapées dans la **console H2** (un SGBD). Ici, on utilise le module `sqlite3` de Python, qui contient un autre SGBD (**SQLite**) : **le langage SQL est le même**, donc chaque requête de ce notebook peut être recopiée telle quelle dans la console H2 (les rares différences sont signalées).

- `execute_fichier("fichier.sql")` joue le rôle de `RUNSCRIPT FROM 'fichier.sql';` dans H2 ;
- `affiche(requete)` exécute une requête et affiche le résultat sous forme de tableau, comme la console H2 ;
- `resultat(requete)` renvoie la liste des enregistrements : elle ne sert qu'aux vérifications.

La requête SQL est écrite entre trois guillemets `""" ... """` : c'est une chaîne de caractères Python qui peut tenir sur plusieurs lignes.

In [1]:
import sqlite3

base = sqlite3.connect(":memory:")   # une base de données vide, en mémoire vive


def execute_fichier(nom_fichier):
    """Équivalent de RUNSCRIPT FROM '...' dans H2 : exécute un fichier .sql."""
    fichier = open(nom_fichier, encoding="utf-8")
    script = fichier.read()
    fichier.close()
    base.executescript(script)


def resultat(requete):
    """Renvoie la liste des enregistrements obtenus (pour les vérifications)."""
    return base.execute(requete).fetchall()


def affiche(requete, maximum=20):
    """Exécute une requête et affiche le résultat comme dans la console H2."""
    curseur = base.execute(requete)
    if curseur.description is None:          # INSERT, UPDATE, DELETE, CREATE...
        print("Instruction exécutée.")
        return
    entete = ""
    for colonne in curseur.description:
        entete = entete + colonne[0] + " | "
    print(entete)
    nombre = 0
    for ligne in curseur:
        if nombre < maximum:
            texte = ""
            for valeur in ligne:
                texte = texte + str(valeur) + " | "
            print(texte)
        nombre = nombre + 1
    if nombre > maximum:
        print("... (seules les " + str(maximum) + " premières lignes sont affichées)")
    print("(" + str(nombre) + " ligne(s))")

---
# QUESTION 1

#### 📝 ÉNONCÉ

> 1. Dans la console H2, recopier et exécuter l'instruction SQL ci-dessous (dans ce langage, la fin d'une instruction est indiquée par un point-virgule, et les indentations sont décoratives).

#### ✅ RÉSOLUTION

```sql
CREATE TABLE Countries (
    id INT,
    code CHAR(2),
    name VARCHAR,
    continent VARCHAR );
```

*Exécution dans le notebook :*

In [2]:
affiche("""
CREATE TABLE Countries (
    id INT,
    code CHAR(2),
    name VARCHAR,
    continent VARCHAR );
""")

Instruction exécutée.


*Vérification (non demandée) :*

In [3]:
affiche("SELECT * FROM Countries")   # la table existe, mais elle est vide

id | code | name | continent | 
(0 ligne(s))


#### 💡 EXPLICATION PÉDAGOGIQUE

- `CREATE TABLE` crée une **relation** (une table) vide : on décrit sa **structure**, pas encore son contenu.
- Pour chaque **attribut** (colonne), on donne son nom et son **domaine de valeurs**, sous la forme d'un type :
  - `INT` : un entier ;
  - `CHAR(2)` : une chaîne d'**au plus 2 caractères** (les codes ISO comme `FR`) ;
  - `VARCHAR` : une chaîne de caractères de longueur quelconque.
- Les attributs sont séparés par des **virgules** (pas de virgule après le dernier), et l'instruction se termine par un **point-virgule**.
- Si on exécute deux fois ce `CREATE TABLE`, H2 répond que la table existe déjà. Pour recommencer : `DROP TABLE Countries;`.

---
# QUESTION 2

> 2. On considère le script SQL ci-dessous.
> ```sql
> INSERT INTO Countries (id, code, name, continent)
>     VALUES (302687, 'FR', 'France', 'EU');
> INSERT INTO Countries (id, code, name, continent)
>     VALUES (302684, 'ES', 'Spain', 'EU');
> SELECT * FROM Countries;
> ```
> Les deux premières instructions permettent d'insérer deux enregistrements (lignes) dans la table. La dernière instruction permet d'afficher la relation (table) Countries.

## 2 (a)

#### 📝 ÉNONCÉ

> (a) Recopier et exécuter ce script.

#### ✅ RÉSOLUTION

```sql
INSERT INTO Countries (id, code, name, continent)
    VALUES (302687, 'FR', 'France', 'EU');
INSERT INTO Countries (id, code, name, continent)
    VALUES (302684, 'ES', 'Spain', 'EU');
SELECT * FROM Countries;
```

*Exécution dans le notebook :*

In [4]:
affiche("""
INSERT INTO Countries (id, code, name, continent)
    VALUES (302687, 'FR', 'France', 'EU');
""")
affiche("""
INSERT INTO Countries (id, code, name, continent)
    VALUES (302684, 'ES', 'Spain', 'EU');
""")
affiche("""
SELECT * FROM Countries;
""")

Instruction exécutée.
Instruction exécutée.
id | code | name | continent | 
302687 | FR | France | EU | 
302684 | ES | Spain | EU | 
(2 ligne(s))


*Vérification (non demandée) :*

In [5]:
assert resultat("SELECT * FROM Countries") == [(302687, "FR", "France", "EU"), (302684, "ES", "Spain", "EU")]

#### 💡 EXPLICATION PÉDAGOGIQUE

- `INSERT INTO Relation (attributs) VALUES (valeurs);` ajoute **un enregistrement**. Les valeurs sont données **dans le même ordre** que les attributs : `302687` va dans `id`, `'FR'` dans `code`, etc.
- Les nombres s'écrivent tels quels, les chaînes entre **guillemets simples**.
- ⚠️ **Le piège du copier-coller depuis le PDF.** Le PDF affiche le code avec des espaces autour des guillemets. En copiant, on obtient `' FR '` au lieu de `'FR'`. Or **tout ce qui est entre les guillemets fait partie de la chaîne**, espaces compris : `' FR '` fait **4 caractères**. H2 refuse alors l'insertion, car l'attribut `code` est de type `CHAR(2)` :

  ```
  Valeur trop longue pour la colonne "CODE CHARACTER(2)": "' FR ' (4)"
  ```

  C'est le SGBD qui **vérifie le domaine de valeurs** : c'est un des services rendus par un SGBD (la cohérence des données). Les espaces **en dehors** des guillemets (autour des virgules, des parenthèses) ne posent aucun problème.
- Remarque : SQLite (utilisé dans ce notebook) est beaucoup moins strict que H2 et **accepte** une chaîne de 4 caractères dans un `CHAR(2)`. C'est une raison de plus pour bien recopier.

## 2 (b)

#### 📝 ÉNONCÉ

> (b) Écrire un script en python qui permet d'enregistrer dans un fichier texte nommé countries.sql l'ensemble des instructions permettant d'ajouter à la relation Countries les données contenues dans le fichier countries.csv.

Le fichier `countries.csv` commence ainsi (une ligne d'en-tête, puis un pays par ligne) :

In [6]:
fichier = open("countries.csv", encoding="utf-8")
for i in range(5):
    print(fichier.readline(), end="")
fichier.close()

"id","code","name","continent"
302672,"AD","Andorra","EU"
302618,"AE","United Arab Emirates","AS"
302619,"AF","Afghanistan","AS"
302722,"AG","Antigua and Barbuda","NA"


#### ✅ RÉSOLUTION

On lit le fichier CSV avec le module `csv` (vu en Première) : `csv.DictReader` donne chaque ligne sous la forme d'un **dictionnaire** dont les clés sont les noms des colonnes. Pour chaque pays, on écrit une instruction `INSERT INTO` dans le fichier `countries.sql`.

In [7]:
import csv


def echappe(texte):
    """Double chaque apostrophe : en SQL, l'apostrophe de Côte d'Ivoire s'écrit ''."""
    resultat = ""
    for caractere in texte:
        if caractere == "'":
            resultat = resultat + "''"
        else:
            resultat = resultat + caractere
    return resultat


fichier_csv = open("countries.csv", encoding="utf-8")
fichier_sql = open("countries.sql", "w", encoding="utf-8")

for pays in csv.DictReader(fichier_csv):
    instruction = ("INSERT INTO Countries (id, code, name, continent) VALUES ("
                   + pays["id"] + ", '"
                   + pays["code"] + "', '"
                   + echappe(pays["name"]) + "', '"
                   + pays["continent"] + "');")
    fichier_sql.write(instruction + "\n")

fichier_csv.close()
fichier_sql.close()

*Vérification (non demandée) :*

In [8]:
fichier = open("countries.sql", encoding="utf-8")
lignes = fichier.readlines()
fichier.close()

print(len(lignes), "instructions écrites. Les trois premières :")
for i in range(3):
    print(lignes[i], end="")

for ligne in lignes:
    if "Ivoire" in ligne:
        print("Avec une apostrophe :", ligne, end="")

assert len(lignes) == 248

248 instructions écrites. Les trois premières :
INSERT INTO Countries (id, code, name, continent) VALUES (302672, 'AD', 'Andorra', 'EU');
INSERT INTO Countries (id, code, name, continent) VALUES (302618, 'AE', 'United Arab Emirates', 'AS');
INSERT INTO Countries (id, code, name, continent) VALUES (302619, 'AF', 'Afghanistan', 'AS');
Avec une apostrophe : INSERT INTO Countries (id, code, name, continent) VALUES (302564, 'CI', 'Côte d''Ivoire', 'AF');


#### 💡 EXPLICATION PÉDAGOGIQUE

- **Le principe :** un fichier `.sql` n'est qu'un fichier texte. On le fabrique en écrivant, ligne par ligne, des chaînes de caractères qui sont des instructions SQL. Le programme Python **génère** du code SQL.
- **Construire l'instruction :** on colle les morceaux avec `+`. Attention aux guillemets : en Python, la chaîne `", '"` contient une virgule, une espace et **un guillemet simple** ; c'est ce guillemet simple qui ouvrira la chaîne SQL `'FR'`. Le nombre `id` n'a pas de guillemets, les chaînes `code`, `name` et `continent` en ont.
- Toutes les valeurs lues dans un CSV sont des **chaînes** (`pays["id"]` vaut `"302687"`), d'où la concaténation directe avec `+`, sans `str()`.
- `"\n"` ajoute un retour à la ligne après chaque instruction (sinon tout serait sur une seule ligne, ce qui marcherait quand même grâce aux points-virgules, mais serait illisible).
- ⚠️ **Le piège : l'apostrophe.** Le nom *Côte d'Ivoire* contient une apostrophe. Sans précaution, on écrirait `'Côte d'Ivoire'` : pour SQL, la chaîne s'arrête après `d`, et la suite `Ivoire'` provoque une **erreur de syntaxe**. En SQL, on **échappe** une apostrophe en la **doublant** : `'Côte d''Ivoire'`. C'est le rôle de la fonction `echappe` (voir la dernière page du polycopié : « l'échappement d'un guillemet s'effectue en le doublant »).
- On ferme les fichiers avec `close()` : c'est à ce moment que tout est réellement écrit sur le disque.

## 2 (c)

#### 📝 ÉNONCÉ

> (c) Dans la console H2, exécuter le contenu du fichier obtenu à l'aide de l'instruction RUNSCRIPT FROM 'chemin/vers/countries.sql';

#### ✅ RÉSOLUTION

Dans la console H2 (avec le chemin du fichier sur l'ordinateur) :

```sql
RUNSCRIPT FROM '/chemin/vers/countries.sql';
SELECT * FROM Countries;
```

Dans le notebook :

In [9]:
execute_fichier("countries.sql")
affiche("SELECT * FROM Countries")

id | code | name | continent | 
302687 | FR | France | EU | 
302684 | ES | Spain | EU | 
302672 | AD | Andorra | EU | 
302618 | AE | United Arab Emirates | AS | 
302619 | AF | Afghanistan | AS | 
302722 | AG | Antigua and Barbuda | NA | 
302723 | AI | Anguilla | NA | 
302673 | AL | Albania | EU | 
302620 | AM | Armenia | AS | 
302556 | AO | Angola | AF | 
302615 | AQ | Antarctica | AN | 
302789 | AR | Argentina | SA | 
302763 | AS | American Samoa | OC | 
302674 | AT | Austria | EU | 
302764 | AU | Australia | OC | 
302725 | AW | Aruba | NA | 
302621 | AZ | Azerbaijan | AS | 
302675 | BA | Bosnia and Herzegovina | EU | 
302726 | BB | Barbados | NA | 
302622 | BD | Bangladesh | AS | 
... (seules les 20 premières lignes sont affichées)
(250 ligne(s))


*Vérification (non demandée) :*

In [10]:
affiche("SELECT COUNT(*) FROM Countries")
affiche("SELECT * FROM Countries WHERE name = 'France'")
affiche("SELECT * FROM Countries WHERE name = 'Côte d''Ivoire'")
assert resultat("SELECT COUNT(*) FROM Countries") == [(250,)]

COUNT(*) | 
250 | 
(1 ligne(s))
id | code | name | continent | 
302687 | FR | France | EU | 
302687 | FR | France | EU | 
(2 ligne(s))
id | code | name | continent | 
302564 | CI | Côte d'Ivoire | AF | 
(1 ligne(s))


#### 💡 EXPLICATION PÉDAGOGIQUE

- `RUNSCRIPT FROM '...'` exécute **toutes** les instructions du fichier, l'une après l'autre. C'est beaucoup plus rapide que de les recopier une par une dans la console.
- Sous Linux, un chemin ressemble à `'/home/utilisateur/Bureau/countries.sql'` ; sous Windows, à `'C:/Users/utilisateur/Desktop/countries.sql'` (avec des `/`).
- **Observation importante :** la table contient **250** enregistrements et non 248. La France et l'Espagne, insérées à la question 2 (a), sont présentes **deux fois** ! Rien n'interdit les doublons, car la table n'a **pas de clé primaire**. Si on avait déclaré `id INT PRIMARY KEY`, le SGBD aurait refusé la deuxième insertion de la France. C'est l'objet de la suite du cours (clé primaire, exercices 4 et 5).
- Dans la requête de vérification, l'apostrophe de *Côte d'Ivoire* est elle aussi doublée.